#Classification Model

This notebook loads features and splits from Notebook 1, trains and compares multiple classifiers (Logistic Regression, Random Forest, XGBoost, LightGBM, and ensemble/stacking variants) using scaffold-grouped cross-validation, performs feature selection, evaluates on a held-out test set, and runs rigorous validation (Y-randomization, repeated scaffold splits, bootstrap CIs, calibration check).


# Environment Setup

In [ ]:
!pip -q install optuna xgboost shap lightgbm


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import wilcoxon, ttest_rel
from sklearn.base import clone, BaseEstimator, ClassifierMixin
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.model_selection import GroupKFold
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    matthews_corrcoef, roc_auc_score, roc_curve, auc,
    confusion_matrix, classification_report, brier_score_loss
)
from sklearn.calibration import calibration_curve # Moved from sklearn.metrics in older versions
from xgboost import XGBClassifier
import xgboost
from lightgbm import LGBMClassifier
import lightgbm
import optuna
import shap
import joblib
from tqdm import tqdm
import sklearn

RANDOM_STATE = 42
TUNE_N_FOLDS = 5
TUNE_N_TRIALS = 100
TRIAL_TIMEOUT = 300
EARLY_STOPPING_PATIENCE = 10
N_YRAND_PERMUTATIONS = 200
N_BOOTSTRAP = 1000

optuna.logging.set_verbosity(optuna.logging.WARNING)

print(f"scikit-learn: {sklearn.__version__} | XGBoost: {xgboost.__version__} | LightGBM: {lightgbm.__version__} | Optuna: {optuna.__version__} | SHAP: {shap.__version__}")

plt.rcParams.update({
    "figure.dpi": 140, "savefig.dpi": 300,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "font.size": 10,
})

# Load `features.pkl` and `splits.pkl`

In [ ]:
features = joblib.load("features.pkl")
splits = joblib.load("splits.pkl")

const = features["constants"]
TIER_TO_LABEL = const["TIER_TO_LABEL"]
LABEL_TO_TIER = const["LABEL_TO_TIER"]
CLASS_LABELS_ORDERED = const["CLASS_LABELS_ORDERED"]
TIER_NAMES_ORDERED = const["TIER_NAMES_ORDERED"]
ACTIVE_PIC50_CUTOFF = const["ACTIVE_PIC50_CUTOFF"]
INACTIVE_PIC50_CUTOFF = const["INACTIVE_PIC50_CUTOFF"]
AD_TANIMOTO_THRESHOLD = const["AD_TANIMOTO_THRESHOLD"]
AD_KNN_K = const["AD_KNN_K"]

hybrid_fp = features["hybrid_fp"]
desc_df_all = features["desc_2d"]
desc_cols = features["desc_cols"]
y_class_full = features["y_class"]
y_pic50_full = features["pIC50"]
smiles_all = features["smiles"]
scaffold_all = features["scaffold"]

X_train, X_val, X_test = splits["X_train"], splits["X_val"], splits["X_test"]
y_train_class, y_val_class, y_test_class = splits["y_train_class"], splits["y_val_class"], splits["y_test_class"]
train_idx, val_idx, test_idx = splits["train_idx"], splits["val_idx"], splits["test_idx"]

print("X_train / X_val / X_test:", X_train.shape, X_val.shape, X_test.shape)
print("Class balance (train):", pd.Series(y_train_class).value_counts().sort_index().to_dict())


# Shared Helpers


In [ ]:
from sklearn.feature_selection import VarianceThreshold

def prepare_filtered_split(train_idx_s, val_idx_s, test_idx_s, variance_threshold=0.01, corr_threshold=0.95):
    fp_cols_s = [f"FP_{i}" for i in range(hybrid_fp.shape[1])]
    X_fp = pd.DataFrame(hybrid_fp, columns=fp_cols_s)
    X_all_s = pd.concat([X_fp, desc_df_all.reset_index(drop=True)], axis=1)

    X_train_raw = X_all_s.iloc[train_idx_s].copy()
    X_val_raw = X_all_s.iloc[val_idx_s].copy()
    X_test_raw = X_all_s.iloc[test_idx_s].copy()

    medians = X_train_raw[desc_cols].median(numeric_only=True).fillna(0.0)
    X_train_raw[desc_cols] = X_train_raw[desc_cols].fillna(medians)
    X_val_raw[desc_cols] = X_val_raw[desc_cols].fillna(medians)
    X_test_raw[desc_cols] = X_test_raw[desc_cols].fillna(medians)

    selector = VarianceThreshold(threshold=variance_threshold)
    X_train_var = pd.DataFrame(selector.fit_transform(X_train_raw), columns=X_train_raw.columns[selector.get_support()])
    X_val_var = pd.DataFrame(selector.transform(X_val_raw), columns=X_train_var.columns)
    X_test_var = pd.DataFrame(selector.transform(X_test_raw), columns=X_train_var.columns)

    corr_matrix = X_train_var.corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [c for c in upper.columns if any(upper[c] > corr_threshold)]

    return X_train_var.drop(columns=to_drop), X_val_var.drop(columns=to_drop), X_test_var.drop(columns=to_drop)

def nearest_training_tanimoto(query_fps, train_fps, batch_size=500):
    query_bool = np.asarray(query_fps).astype(np.int32)
    train_bool = np.asarray(train_fps).astype(np.int32)
    train_popcount = train_bool.sum(axis=1)
    nearest = np.empty(len(query_bool), dtype=float)
    for start in range(0, len(query_bool), batch_size):
        batch = query_bool[start:start + batch_size]
        query_popcount = batch.sum(axis=1)
        intersection = batch @ train_bool.T
        union = query_popcount[:, None] + train_popcount[None, :] - intersection
        similarity = np.divide(intersection, union, out=np.zeros_like(intersection, dtype=float), where=union != 0)
        nearest[start:start + batch_size] = similarity.max(axis=1)
    return nearest

def topk_mean_training_tanimoto(query_fps, train_fps, k=5, batch_size=500):
    query_bool = np.asarray(query_fps).astype(np.int32)
    train_bool = np.asarray(train_fps).astype(np.int32)
    train_popcount = train_bool.sum(axis=1)
    k_eff = min(k, len(train_bool))
    topk_mean = np.empty(len(query_bool), dtype=float)
    for start in range(0, len(query_bool), batch_size):
        batch = query_bool[start:start + batch_size]
        query_popcount = batch.sum(axis=1)
        intersection = batch @ train_bool.T
        union = query_popcount[:, None] + train_popcount[None, :] - intersection
        similarity = np.divide(intersection, union, out=np.zeros_like(intersection, dtype=float), where=union != 0)
        top_k_vals = -np.partition(-similarity, k_eff - 1, axis=1)[:, :k_eff]
        topk_mean[start:start + batch_size] = top_k_vals.mean(axis=1)
    return topk_mean


# Baseline: Logistic Regression

In [ ]:
logreg_baseline = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=5000, class_weight="balanced", random_state=RANDOM_STATE),
)
logreg_baseline.fit(X_train, y_train_class)

logreg_val_pred = logreg_baseline.predict(X_val)
logreg_test_pred = logreg_baseline.predict(X_test)
logreg_test_proba = logreg_baseline.predict_proba(X_test)

logreg_metrics = {
    "Accuracy": accuracy_score(y_test_class, logreg_test_pred),
    "Precision (macro)": precision_score(y_test_class, logreg_test_pred, average="macro", zero_division=0),
    "Recall (macro)": recall_score(y_test_class, logreg_test_pred, average="macro", zero_division=0),
    "F1 (macro)": f1_score(y_test_class, logreg_test_pred, average="macro", zero_division=0),
    "MCC": matthews_corrcoef(y_test_class, logreg_test_pred),
    "ROC-AUC (macro OVR)": roc_auc_score(y_test_class, logreg_test_proba, multi_class="ovr", average="macro"),
}
print("Logistic Regression test metrics:")
for k,v in logreg_metrics.items():
    print(f"{k:20s}: {v:.4f}")


# Scaffold-Grouped CV Folds (5-fold) for Optuna

In [ ]:
tune_idx = np.concatenate([train_idx, val_idx])
tune_scaffold_groups = scaffold_all[tune_idx]

gkf = GroupKFold(n_splits=TUNE_N_FOLDS)
cv_folds = []
for fold_i, (fold_train_pos, fold_val_pos) in enumerate(gkf.split(tune_idx, groups=tune_scaffold_groups)):
    fold_train_idx = tune_idx[fold_train_pos]
    fold_val_idx = tune_idx[fold_val_pos]
    X_ft, X_fv, _ = prepare_filtered_split(fold_train_idx, fold_val_idx, fold_val_idx)

    cv_folds.append({
        "X_train": X_ft, "X_val": X_fv,
        "y_train_class": y_class_full[fold_train_idx],
        "y_val_class": y_class_full[fold_val_idx],
        "sample_weight_train": compute_sample_weight("balanced", y_class_full[fold_train_idx]),
    })
    n_shared = len(set(scaffold_all[fold_train_idx]) & set(scaffold_all[fold_val_idx]))
    print(f"  Fold {fold_i}: {len(fold_train_idx)} train / {len(fold_val_idx)} val, "
          f"{X_ft.shape[1]} features, {n_shared} scaffolds shared (should be 0)")

print(f"\nBuilt {TUNE_N_FOLDS} scaffold-grouped CV folds. Test set ({len(test_idx)}) untouched.")


In [ ]:


FEATURE_COUNT_CHOICES = [3000, 1500, 750, 400, 200, 100]

def _rank_fold_features(X_tr, y_tr):
    scout = RandomForestClassifier(
        n_estimators=300, max_depth=None, class_weight="balanced",
        random_state=RANDOM_STATE, n_jobs=-1,
    )
    scout.fit(X_tr, y_tr)
    order = np.argsort(scout.feature_importances_)[::-1]
    return list(X_tr.columns[order])

for fold in cv_folds:
    fold["feature_rank"] = _rank_fold_features(fold["X_train"], fold["y_train_class"])

print("Per-fold feature rankings computed (train-partition only, no leakage into fold validation).")
for i, fold in enumerate(cv_folds):
    print(f"  Fold {i}: {len(fold['feature_rank'])} candidate features ranked")

def top_fold_cols(fold, n_features):
    n_take = min(n_features, len(fold["feature_rank"]))
    return fold["feature_rank"][:n_take]

# Random Forest (Optuna-Tuned)

In [ ]:


import multiprocessing as mp

def fit_and_score_rf(params):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        model_params = {k: v for k, v in params.items() if k != "n_features"}
        n_features = params["n_features"]
        fold_scores = []
        for fold in cv_folds:
            cols = top_fold_cols(fold, n_features)
            model = RandomForestClassifier(**model_params, random_state=RANDOM_STATE, n_jobs=-1)
            model.fit(fold["X_train"][cols], fold["y_train_class"])
            fold_scores.append(matthews_corrcoef(fold["y_val_class"], model.predict(fold["X_val"][cols])))
    return float(np.mean(fold_scores))

def objective_rf(trial):
    bootstrap = trial.suggest_categorical("bootstrap", [True, False])
    class_weight = trial.suggest_categorical("class_weight", ["balanced", "balanced_subsample", None])
    if not bootstrap and class_weight == "balanced_subsample":
        class_weight = None
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 300, 1000),
        "max_depth": trial.suggest_int("max_depth", 5, 40),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
        "bootstrap": bootstrap,
        "class_weight": class_weight,
        "n_features": trial.suggest_categorical("n_features", FEATURE_COUNT_CHOICES),
    }
    with mp.Pool(1) as pool:
        result = pool.apply_async(fit_and_score_rf, (params,))
        try:
            return result.get(timeout=TRIAL_TIMEOUT)
        except Exception:
            return -999.0

study_rf = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))

print("Optimizing Random Forest with 5-fold CV (nested feature selection)...")
with tqdm(total=TUNE_N_TRIALS, desc="RF trials") as pbar:
    def callback(study, trial):
        pbar.update(1)
        pbar.set_postfix({
            'Best': f'{study.best_value:.4f}',
            'Current': f'{trial.value:.4f}',
            'Depth': trial.params.get('max_depth', 0),
            'Feats': trial.params.get('n_features', 0),
        })
        if study.best_trial and (trial.number - study.best_trial.number) > EARLY_STOPPING_PATIENCE:
            study.stop()
    study_rf.optimize(objective_rf, n_trials=TUNE_N_TRIALS, callbacks=[callback])

print("Best RF params:", study_rf.best_params)
print("Best RF CV MCC:", study_rf.best_value)

rf_best_params = {k: v for k, v in study_rf.best_params.items() if k != "n_features"}
rf_n_features = study_rf.best_params["n_features"]

rf_full_rank = _rank_fold_features(X_train, y_train_class)   # ranked on train_idx only
rf_selected_cols = rf_full_rank[:min(rf_n_features, len(rf_full_rank))]
print(f"RF selected feature count: {len(rf_selected_cols)} (Optuna-tuned n_features={rf_n_features})")

best_rf = RandomForestClassifier(**rf_best_params, random_state=RANDOM_STATE, n_jobs=-1)
best_rf.fit(X_train[rf_selected_cols], y_train_class)
rf_val_pred = best_rf.predict(X_val[rf_selected_cols])
rf_test_pred = best_rf.predict(X_test[rf_selected_cols])
rf_test_proba = best_rf.predict_proba(X_test[rf_selected_cols])

rf_metrics = {
    "Accuracy": accuracy_score(y_test_class, rf_test_pred),
    "Precision (macro)": precision_score(y_test_class, rf_test_pred, average="macro", zero_division=0),
    "Recall (macro)": recall_score(y_test_class, rf_test_pred, average="macro", zero_division=0),
    "F1 (macro)": f1_score(y_test_class, rf_test_pred, average="macro", zero_division=0),
    "MCC": matthews_corrcoef(y_test_class, rf_test_pred),
    "ROC-AUC (macro OVR)": roc_auc_score(y_test_class, rf_test_proba, multi_class="ovr", average="macro"),
}
print("\nRandom Forest test metrics:")
for k,v in rf_metrics.items():
    print(f"{k:20s}: {v:.4f}")

# XGBoost (Optuna-Tuned, grid-search threshold tuning)

In [ ]:
"""# XGBoost (Optuna-Tuned, nested feature selection, grid-search threshold tuning)"""

def objective_xgb(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 300, 1500),
        "max_depth": trial.suggest_int("max_depth", 2, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight": trial.suggest_float("min_child_weight", 1, 20),
        "gamma": trial.suggest_float("gamma", 0, 10),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 20.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 20.0, log=True),
        "objective": "multi:softprob",
        "num_class": 3,
        "eval_metric": "mlogloss",
        "tree_method": "hist",  # change to 'gpu_hist' for GPU acceleration
        "random_state": RANDOM_STATE,
        "n_jobs": -1,
        "early_stopping_rounds": 50,
    }
    n_features = trial.suggest_categorical("n_features", FEATURE_COUNT_CHOICES)
    fold_scores = []
    for fold in cv_folds:
        cols = top_fold_cols(fold, n_features)
        model = XGBClassifier(**params)
        model.fit(
            fold["X_train"][cols], fold["y_train_class"],
            sample_weight=fold["sample_weight_train"],
            eval_set=[(fold["X_val"][cols], fold["y_val_class"])],
            verbose=False,
        )
        fold_scores.append(matthews_corrcoef(fold["y_val_class"], model.predict(fold["X_val"][cols])))
    return float(np.mean(fold_scores))

study_xgb = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))

print("Optimizing XGBoost with 5-fold CV (nested feature selection)...")
with tqdm(total=TUNE_N_TRIALS, desc="XGB trials") as pbar:
    def callback(study, trial):
        pbar.update(1)
        pbar.set_postfix({
            'Best': f'{study.best_value:.4f}',
            'Current': f'{trial.value:.4f}',
            'Depth': trial.params.get('max_depth', 0),
            'Feats': trial.params.get('n_features', 0),
        })
        if study.best_trial and (trial.number - study.best_trial.number) > EARLY_STOPPING_PATIENCE:
            study.stop()
    study_xgb.optimize(objective_xgb, n_trials=TUNE_N_TRIALS, callbacks=[callback])

print("Best XGB params:", study_xgb.best_params)
print("Best XGB CV MCC:", study_xgb.best_value)

xgb_best_params = {k: v for k, v in study_xgb.best_params.items() if k != "n_features"}
xgb_n_features = study_xgb.best_params["n_features"]

xgb_full_rank = _rank_fold_features(X_train, y_train_class)
xgb_selected_cols = xgb_full_rank[:min(xgb_n_features, len(xgb_full_rank))]
print(f"XGB selected feature count: {len(xgb_selected_cols)} (Optuna-tuned n_features={xgb_n_features})")

sample_weight_train = compute_sample_weight("balanced", y_train_class)
best_xgb = XGBClassifier(
    **xgb_best_params,
    objective="multi:softprob", num_class=3,
    eval_metric="mlogloss", tree_method="hist",  # change to 'gpu_hist' for GPU
    random_state=RANDOM_STATE, n_jobs=-1,
    early_stopping_rounds=50,
)
best_xgb.fit(
    X_train[xgb_selected_cols], y_train_class,
    sample_weight=sample_weight_train,
    eval_set=[(X_val[xgb_selected_cols], y_val_class)],
    verbose=False,
)
print(f"Best iteration: {best_xgb.best_iteration}")

xgb_val_proba = best_xgb.predict_proba(X_val[xgb_selected_cols])
xgb_test_proba = best_xgb.predict_proba(X_test[xgb_selected_cols])
xgb_val_pred_raw = np.argmax(xgb_val_proba, axis=1)
xgb_test_pred_raw = np.argmax(xgb_test_proba, axis=1)

print("\nTuning decision thresholds via grid search...")
best_val_mcc = -1
best_bias = np.zeros(3)
bias_range = np.linspace(-2.0, 2.0, 11)
for b0 in bias_range:
    for b1 in bias_range:
        for b2 in bias_range:
            bias = np.array([b0, b1, b2])
            adjusted = xgb_val_proba * np.exp(bias)
            preds = np.argmax(adjusted, axis=1)
            mcc = matthews_corrcoef(y_val_class, preds)
            if mcc > best_val_mcc:
                best_val_mcc = mcc
                best_bias = bias
print(f"Best validation MCC after threshold tuning: {best_val_mcc:.4f}")
print(f"Best bias (log-space): {best_bias}")

xgb_test_pred = np.argmax(xgb_test_proba * np.exp(best_bias), axis=1)

xgb_metrics = {
    "Accuracy": accuracy_score(y_test_class, xgb_test_pred),
    "Precision (macro)": precision_score(y_test_class, xgb_test_pred, average="macro", zero_division=0),
    "Recall (macro)": recall_score(y_test_class, xgb_test_pred, average="macro", zero_division=0),
    "F1 (macro)": f1_score(y_test_class, xgb_test_pred, average="macro", zero_division=0),
    "MCC": matthews_corrcoef(y_test_class, xgb_test_pred),
    "ROC-AUC (macro OVR)": roc_auc_score(y_test_class, xgb_test_proba, multi_class="ovr", average="macro"),
}
print("\nXGBoost test metrics (threshold-tuned):")
for k,v in xgb_metrics.items():
    print(f"{k:20s}: {v:.4f}")

# LightGBM (Optuna-Tuned)

In [ ]:
"""# LightGBM (Optuna-Tuned, nested feature selection)"""

import optuna
import lightgbm as lgb
import numpy as np
from sklearn.metrics import matthews_corrcoef, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.utils.class_weight import compute_sample_weight
from tqdm import tqdm

def objective_lgb(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 300, 1500),
        "max_depth": trial.suggest_int("max_depth", 2, 15),
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight": trial.suggest_float("min_child_weight", 1, 20),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 20.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 20.0, log=True),
        "objective": "multiclass",
        "num_class": 3,
        "random_state": RANDOM_STATE,
        "n_jobs": -1,
        "verbosity": -1,
    }
    n_features = trial.suggest_categorical("n_features", FEATURE_COUNT_CHOICES)
    fold_scores = []
    for fold in cv_folds:
        cols = top_fold_cols(fold, n_features)
        model = lgb.LGBMClassifier(**params)
        model.fit(
            fold["X_train"][cols], fold["y_train_class"],
            sample_weight=fold["sample_weight_train"],
        )
        fold_scores.append(matthews_corrcoef(fold["y_val_class"], model.predict(fold["X_val"][cols])))
    return float(np.mean(fold_scores))

study_lgb = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
print("Optimizing LightGBM with 5-fold CV (nested feature selection)...")

with tqdm(total=TUNE_N_TRIALS, desc="LGB trials", unit="trial") as pbar:
    def callback(study, trial):
        pbar.update(1)
        pbar.set_postfix({
            'Best': f'{study.best_value:.4f}',
            'Current': f'{trial.value:.4f}',
            'Depth': trial.params.get('max_depth', 0),
            'Feats': trial.params.get('n_features', 0),
        })
        if study.best_trial and (trial.number - study.best_trial.number) > EARLY_STOPPING_PATIENCE:
            study.stop()
    study_lgb.optimize(objective_lgb, n_trials=TUNE_N_TRIALS, callbacks=[callback])

print("Best LGB params:", study_lgb.best_params)
print("Best LGB CV MCC:", study_lgb.best_value)

lgb_best_params = {k: v for k, v in study_lgb.best_params.items() if k != "n_features"}
lgb_n_features = study_lgb.best_params["n_features"]

lgb_full_rank = _rank_fold_features(X_train, y_train_class)
lgb_selected_cols = lgb_full_rank[:min(lgb_n_features, len(lgb_full_rank))]
print(f"LGB selected feature count: {len(lgb_selected_cols)} (Optuna-tuned n_features={lgb_n_features})")

best_lgb = lgb.LGBMClassifier(
    **lgb_best_params,
    objective="multiclass",
    num_class=3,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=-1
)
best_lgb.fit(X_train[lgb_selected_cols], y_train_class, sample_weight=compute_sample_weight("balanced", y_train_class))

lgb_val_proba = best_lgb.predict_proba(X_val[lgb_selected_cols])
lgb_val_pred = best_lgb.predict(X_val[lgb_selected_cols])
lgb_test_proba = best_lgb.predict_proba(X_test[lgb_selected_cols])

best_val_mcc_lgb = -1
best_bias_lgb = np.zeros(3)
for b0 in bias_range:
    for b1 in bias_range:
        for b2 in bias_range:
            bias = np.array([b0, b1, b2])
            adjusted = lgb_val_proba * np.exp(bias)
            preds = np.argmax(adjusted, axis=1)
            mcc = matthews_corrcoef(y_val_class, preds)
            if mcc > best_val_mcc_lgb:
                best_val_mcc_lgb = mcc
                best_bias_lgb = bias

print(f"Best LGB validation MCC after threshold tuning: {best_val_mcc_lgb:.4f}")
lgb_test_pred_tuned = np.argmax(lgb_test_proba * np.exp(best_bias_lgb), axis=1)

lgb_metrics = {
    "Accuracy": accuracy_score(y_test_class, lgb_test_pred_tuned),
    "Precision (macro)": precision_score(y_test_class, lgb_test_pred_tuned, average="macro", zero_division=0),
    "Recall (macro)": recall_score(y_test_class, lgb_test_pred_tuned, average="macro", zero_division=0),
    "F1 (macro)": f1_score(y_test_class, lgb_test_pred_tuned, average="macro", zero_division=0),
    "MCC": matthews_corrcoef(y_test_class, lgb_test_pred_tuned),
    "ROC-AUC (macro OVR)": roc_auc_score(y_test_class, lgb_test_proba, multi_class="ovr", average="macro"),
}
print("\nLightGBM test metrics (threshold-tuned):")
for k, v in lgb_metrics.items():
    print(f"{k:20s}: {v:.4f}")

# Ensemble and Stacking (RF + XGB)


In [ ]:
class ColumnSubsetWrapper(BaseEstimator, ClassifierMixin):
    """Wraps a fitted-hyperparameter estimator and restricts it to the exact
    feature columns it was actually tuned on. Needed because plain clone()
    only copies hyperparameters, not which columns the model was tuned for --
    passing it the full X silently evaluates it on a feature space it was
    never optimized against."""
    def __init__(self, estimator=None, cols=None):
        self.estimator = estimator
        self.cols = cols

    def _select(self, X):
        return X[self.cols] if self.cols is not None else X

    def fit(self, X, y, sample_weight=None):
        self.estimator_ = clone(self.estimator)
        X_sub = self._select(X)
        if sample_weight is not None:
            self.estimator_.fit(X_sub, y, sample_weight=sample_weight)
        else:
            self.estimator_.fit(X_sub, y)
        self.classes_ = self.estimator_.classes_
        return self

    def predict(self, X):
        return self.estimator_.predict(self._select(X))

    def predict_proba(self, X):
        return self.estimator_.predict_proba(self._select(X))


class ProbaAverageEnsembleClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, estimator_a=None, estimator_b=None):
        self.estimator_a = estimator_a
        self.estimator_b = estimator_b

    def fit(self, X, y, sample_weight=None):
        self.estimator_a_ = clone(self.estimator_a)
        self.estimator_b_ = clone(self.estimator_b)
        if sample_weight is not None:
            self.estimator_a_.fit(X, y, sample_weight=sample_weight)
            self.estimator_b_.fit(X, y, sample_weight=sample_weight)
        else:
            self.estimator_a_.fit(X, y)
            self.estimator_b_.fit(X, y)
        self.classes_ = self.estimator_a_.classes_
        return self

    def predict_proba(self, X):
        return (self.estimator_a_.predict_proba(X) + self.estimator_b_.predict_proba(X)) / 2.0

    def predict(self, X):
        return self.classes_[np.argmax(self.predict_proba(X), axis=1)]


rf_for_ensemble = clone(best_rf)
rf_for_ensemble.set_params(class_weight=None)
xgb_for_ensemble = clone(best_xgb)
xgb_for_ensemble.set_params(early_stopping_rounds=None)


ensemble_clf = ProbaAverageEnsembleClassifier(
    estimator_a=ColumnSubsetWrapper(rf_for_ensemble, cols=rf_selected_cols),
    estimator_b=ColumnSubsetWrapper(xgb_for_ensemble, cols=xgb_selected_cols),
)
ensemble_clf.fit(X_train, y_train_class, sample_weight=compute_sample_weight("balanced", y_train_class))
ensemble_val_pred = ensemble_clf.predict(X_val)
ensemble_test_pred = ensemble_clf.predict(X_test)
ensemble_test_proba = ensemble_clf.predict_proba(X_test)

ensemble_metrics = {
    "Accuracy": accuracy_score(y_test_class, ensemble_test_pred),
    "Precision (macro)": precision_score(y_test_class, ensemble_test_pred, average="macro", zero_division=0),
    "Recall (macro)": recall_score(y_test_class, ensemble_test_pred, average="macro", zero_division=0),
    "F1 (macro)": f1_score(y_test_class, ensemble_test_pred, average="macro", zero_division=0),
    "MCC": matthews_corrcoef(y_test_class, ensemble_test_pred),
    "ROC-AUC (macro OVR)": roc_auc_score(y_test_class, ensemble_test_proba, multi_class="ovr", average="macro"),
}
print("Ensemble test metrics:")
for k,v in ensemble_metrics.items():
    print(f"{k:20s}: {v:.4f}")


rf_for_stacking = clone(best_rf)
rf_for_stacking.set_params(class_weight=None)
xgb_for_stacking = clone(best_xgb)
xgb_for_stacking.set_params(early_stopping_rounds=None)


stacking_clf = StackingClassifier(
    estimators=[
        ("rf", ColumnSubsetWrapper(rf_for_stacking, cols=rf_selected_cols)),
        ("xgb", ColumnSubsetWrapper(xgb_for_stacking, cols=xgb_selected_cols)),
    ],
    final_estimator=LogisticRegression(max_iter=5000, class_weight=None, random_state=RANDOM_STATE),
    stack_method="predict_proba", passthrough=False, cv=3, n_jobs=-1,
)
stacking_clf.fit(X_train, y_train_class, sample_weight=compute_sample_weight("balanced", y_train_class))
stacking_val_pred = stacking_clf.predict(X_val)
stacking_test_pred = stacking_clf.predict(X_test)
stacking_test_proba = stacking_clf.predict_proba(X_test)

stacking_metrics = {
    "Accuracy": accuracy_score(y_test_class, stacking_test_pred),
    "Precision (macro)": precision_score(y_test_class, stacking_test_pred, average="macro", zero_division=0),
    "Recall (macro)": recall_score(y_test_class, stacking_test_pred, average="macro", zero_division=0),
    "F1 (macro)": f1_score(y_test_class, stacking_test_pred, average="macro", zero_division=0),
    "MCC": matthews_corrcoef(y_test_class, stacking_test_pred),
    "ROC-AUC (macro OVR)": roc_auc_score(y_test_class, stacking_test_proba, multi_class="ovr", average="macro"),
}
print("\nStacking test metrics:")
for k,v in stacking_metrics.items():
    print(f"{k:20s}: {v:.4f}")

# Model Comparison and Selection (on Validation MCC)

We select the model with the highest validation MCC.

In [ ]:
classifier_candidates = {
    "Logistic Regression": logreg_baseline,
    "Random Forest Classifier": best_rf,
    "XGBoost Classifier": best_xgb,
    "LightGBM Classifier": best_lgb,
    "Ensemble (RF + XGB)": ensemble_clf,
    "Stacking (RF + XGB -> LR)": stacking_clf,
}

classifier_val_preds = {
    "Logistic Regression": logreg_val_pred,
    "Random Forest Classifier": rf_val_pred,
    "XGBoost Classifier": xgb_val_pred_raw,
    "LightGBM Classifier": lgb_val_pred,
    "Ensemble (RF + XGB)": ensemble_val_pred,
    "Stacking (RF + XGB -> LR)": stacking_val_pred,
}

xgb_val_pred_tuned = np.argmax(xgb_val_proba * np.exp(best_bias), axis=1)
lgb_val_pred_tuned = np.argmax(lgb_val_proba * np.exp(best_bias_lgb), axis=1)  # was: best_lgb.predict_proba(X_val) -> wrong feature set, shape mismatch
classifier_val_preds["XGBoost Classifier"] = xgb_val_pred_tuned
classifier_val_preds["LightGBM Classifier"] = lgb_val_pred_tuned

validation_selection = pd.DataFrame([
    {
        "Model": name,
        "Validation Accuracy": accuracy_score(y_val_class, classifier_val_preds[name]),
        "Validation F1 (macro)": f1_score(y_val_class, classifier_val_preds[name], average="macro", zero_division=0),
        "Validation MCC": matthews_corrcoef(y_val_class, classifier_val_preds[name]),
    }
    for name in classifier_candidates
])

best_model_name = validation_selection.sort_values("Validation MCC", ascending=False).iloc[0]["Model"]
best_model = classifier_candidates[best_model_name]

print("Validation set performance:\n", validation_selection.to_string(index=False))
print(f"\nSelected best model: {best_model_name} (Validation MCC = {validation_selection[validation_selection['Model']==best_model_name]['Validation MCC'].values[0]:.4f})")

# Feature Reduction: Eliminate Low-Importance Features



In [ ]:
MODEL_SELECTED_COLS = {
    "Random Forest Classifier": rf_selected_cols,
    "XGBoost Classifier": xgb_selected_cols,
    "LightGBM Classifier": lgb_selected_cols,
}

if best_model_name in MODEL_SELECTED_COLS:
    reduced_cols = MODEL_SELECTED_COLS[best_model_name]
    print(f"Using {len(reduced_cols)} features selected by {best_model_name}'s nested Optuna search.")

    final_train_X = pd.concat([X_train, X_val])[reduced_cols]
    final_train_y = np.concatenate([y_train_class, y_val_class])
    final_train_idx = tune_idx
    final_test_X = X_test[reduced_cols]
    final_sample_weight = compute_sample_weight("balanced", final_train_y)

    def make_fresh_model():
        if best_model_name == "XGBoost Classifier":
            return XGBClassifier(**xgb_best_params, objective="multi:softprob", num_class=3,
                                  eval_metric="mlogloss", tree_method="hist", random_state=RANDOM_STATE, n_jobs=-1)
        elif best_model_name == "Random Forest Classifier":
            return RandomForestClassifier(**rf_best_params, random_state=RANDOM_STATE, n_jobs=-1)
        elif best_model_name == "LightGBM Classifier":
            return LGBMClassifier(**lgb_best_params, objective="multiclass", num_class=3, random_state=RANDOM_STATE, n_jobs=-1)

    def fit_fresh(model, X, y, sw=None):
        if best_model_name == "Random Forest Classifier":
            model.fit(X, y)
        else:
            model.fit(X, y, sample_weight=sw)
        return model

    final_bias = None
    if best_model_name in ["XGBoost Classifier", "LightGBM Classifier"]:
        print("\nRe-tuning decision threshold via scaffold-grouped out-of-fold predictions...")
        oof_groups = scaffold_all[final_train_idx]
        gkf_oof = GroupKFold(n_splits=TUNE_N_FOLDS)
        oof_proba = np.zeros((len(final_train_y), 3))
        for fold_tr_pos, fold_va_pos in gkf_oof.split(final_train_X, final_train_y, groups=oof_groups):
            m = make_fresh_model()
            fit_fresh(m, final_train_X.iloc[fold_tr_pos], final_train_y[fold_tr_pos],
                      sw=final_sample_weight[fold_tr_pos])
            oof_proba[fold_va_pos] = m.predict_proba(final_train_X.iloc[fold_va_pos])


        def grid_search_bias_2d(proba, y_true, half_width, n_points=21):
            search_range = np.linspace(-half_width, half_width, n_points)  # includes 0
            best_mcc = -1
            best_b0b1 = (0.0, 0.0)
            for b0 in search_range:
                for b1 in search_range:
                    bias = np.array([b0, b1, 0.0])
                    preds = np.argmax(proba * np.exp(bias), axis=1)
                    mcc = matthews_corrcoef(y_true, preds)
                    if mcc > best_mcc:
                        best_mcc = mcc
                        best_b0b1 = (b0, b1)
            return best_b0b1, best_mcc

        MAX_HALF_WIDTH = 12.0
        half_width = 4.0
        while True:
            (b0, b1), best_val_mcc_final = grid_search_bias_2d(oof_proba, final_train_y, half_width)
            on_boundary = np.isclose(abs(b0), half_width) or np.isclose(abs(b1), half_width)
            if not on_boundary or half_width >= MAX_HALF_WIDTH:
                if on_boundary:
                    print(f"  [warn] bias still on boundary at half_width={half_width} (hit MAX_HALF_WIDTH cap); "
                          f"using this result but treat it as provisional.")
                break
            print(f"  Bias hit boundary at half_width={half_width} (b0={b0}, b1={b1}); widening and re-searching...")
            half_width *= 2

        final_bias = np.array([b0, b1, 0.0])
        no_correction_mcc = matthews_corrcoef(final_train_y, np.argmax(oof_proba, axis=1))
        print(f"Re-tuned out-of-fold MCC: {best_val_mcc_final:.4f}, bias: {final_bias} "
              f"(search half-width used: {half_width})")
        print(f"For comparison, out-of-fold MCC with NO threshold correction (plain argmax): "
              f"{no_correction_mcc:.4f}")
        if np.isclose(best_val_mcc_final, no_correction_mcc):
            print("  -> Threshold tuning did not improve on the default decision rule; "
                  "consider reporting final_model with no bias correction for simplicity.")

    final_model = make_fresh_model()
    fit_fresh(final_model, final_train_X, final_train_y, sw=final_sample_weight)
else:
    print(f"'{best_model_name}' does not use nested feature selection; using its features as-is.")
    final_model = best_model
    final_train_X = X_train
    final_train_y = y_train_class
    final_train_idx = train_idx
    final_test_X = X_test
    final_bias = None

if final_bias is not None:
    test_proba = final_model.predict_proba(final_test_X)
    test_pred = np.argmax(test_proba * np.exp(final_bias), axis=1)
else:
    test_pred = final_model.predict(final_test_X)

final_test_metrics = {
    "Accuracy": accuracy_score(y_test_class, test_pred),
    "Precision (macro)": precision_score(y_test_class, test_pred, average="macro", zero_division=0),
    "Recall (macro)": recall_score(y_test_class, test_pred, average="macro", zero_division=0),
    "F1 (macro)": f1_score(y_test_class, test_pred, average="macro", zero_division=0),
    "MCC": matthews_corrcoef(y_test_class, test_pred),
}
print("\nFinal model test metrics:")
for k,v in final_test_metrics.items():
    print(f"{k:20s}: {v:.4f}")

In [ ]:
BIAS_IMPROVEMENT_MARGIN = 0.01  # OOF MCC must improve by at least this much to justify using bias

MODEL_SELECTED_COLS = {
    "Random Forest Classifier": rf_selected_cols,
    "XGBoost Classifier": xgb_selected_cols,
    "LightGBM Classifier": lgb_selected_cols,
}

if best_model_name in MODEL_SELECTED_COLS:
    reduced_cols = MODEL_SELECTED_COLS[best_model_name]
    print(f"Using {len(reduced_cols)} features selected by {best_model_name}'s nested Optuna search.")

    final_train_X = pd.concat([X_train, X_val])[reduced_cols]
    final_train_y = np.concatenate([y_train_class, y_val_class])
    final_train_idx = tune_idx
    final_test_X = X_test[reduced_cols]
    final_sample_weight = compute_sample_weight("balanced", final_train_y)

    def make_fresh_model():
        if best_model_name == "XGBoost Classifier":
            return XGBClassifier(**xgb_best_params, objective="multi:softprob", num_class=3,
                                  eval_metric="mlogloss", tree_method="hist", random_state=RANDOM_STATE, n_jobs=-1)
        elif best_model_name == "Random Forest Classifier":
            return RandomForestClassifier(**rf_best_params, random_state=RANDOM_STATE, n_jobs=-1)
        elif best_model_name == "LightGBM Classifier":
            return LGBMClassifier(**lgb_best_params, objective="multiclass", num_class=3, random_state=RANDOM_STATE, n_jobs=-1)

    def fit_fresh(model, X, y, sw=None):
        if best_model_name == "Random Forest Classifier":
            model.fit(X, y)
        else:
            model.fit(X, y, sample_weight=sw)
        return model

    final_bias = None
    if best_model_name in ["XGBoost Classifier", "LightGBM Classifier"]:
        print("\nComputing scaffold-grouped out-of-fold predictions...")
        oof_groups = scaffold_all[final_train_idx]
        gkf_oof = GroupKFold(n_splits=TUNE_N_FOLDS)
        oof_proba = np.zeros((len(final_train_y), 3))
        for fold_tr_pos, fold_va_pos in gkf_oof.split(final_train_X, final_train_y, groups=oof_groups):
            m = make_fresh_model()
            fit_fresh(m, final_train_X.iloc[fold_tr_pos], final_train_y[fold_tr_pos],
                      sw=final_sample_weight[fold_tr_pos])
            oof_proba[fold_va_pos] = m.predict_proba(final_train_X.iloc[fold_va_pos])

        no_correction_mcc = matthews_corrcoef(final_train_y, np.argmax(oof_proba, axis=1))

        # 2D search (class 2 fixed at 0 -- only relative offsets matter, see prior bug fix).
        def grid_search_bias_2d(proba, y_true, half_width, n_points=21):
            r = np.linspace(-half_width, half_width, n_points)
            best_mcc, best_b0b1 = -1, (0.0, 0.0)
            for b0 in r:
                for b1 in r:
                    preds = np.argmax(proba * np.exp([b0, b1, 0.0]), axis=1)
                    mcc = matthews_corrcoef(y_true, preds)
                    if mcc > best_mcc:
                        best_mcc, best_b0b1 = mcc, (b0, b1)
            return best_b0b1, best_mcc

        half_width = 4.0
        while True:
            (b0, b1), tuned_mcc = grid_search_bias_2d(oof_proba, final_train_y, half_width)
            if not (np.isclose(abs(b0), half_width) or np.isclose(abs(b1), half_width)) or half_width >= 12.0:
                break
            half_width *= 2

        improvement = tuned_mcc - no_correction_mcc
        print(f"OOF MCC -- no correction: {no_correction_mcc:.4f} | tuned (bias=[{b0:.2f},{b1:.2f},0.00])]: "
              f"{tuned_mcc:.4f} | improvement: {improvement:+.4f}")

        # PRE-COMMITTED DECISION RULE (test set not consulted):
        if improvement >= BIAS_IMPROVEMENT_MARGIN:
            final_bias = np.array([b0, b1, 0.0])
            print(f"  -> Improvement >= margin ({BIAS_IMPROVEMENT_MARGIN}); using bias correction.")
        else:
            final_bias = None
            print(f"  -> Improvement < margin ({BIAS_IMPROVEMENT_MARGIN}); using plain argmax (no correction).")

    final_model = make_fresh_model()
    fit_fresh(final_model, final_train_X, final_train_y, sw=final_sample_weight)
else:
    print(f"'{best_model_name}' does not use nested feature selection; using its features as-is.")
    final_model = best_model
    final_train_X = X_train
    final_train_y = y_train_class
    final_train_idx = train_idx
    final_test_X = X_test
    final_bias = None

if final_bias is not None:
    test_proba = final_model.predict_proba(final_test_X)
    test_pred = np.argmax(test_proba * np.exp(final_bias), axis=1)
else:
    test_pred = final_model.predict(final_test_X)

final_test_metrics = {
    "Accuracy": accuracy_score(y_test_class, test_pred),
    "Precision (macro)": precision_score(y_test_class, test_pred, average="macro", zero_division=0),
    "Recall (macro)": recall_score(y_test_class, test_pred, average="macro", zero_division=0),
    "F1 (macro)": f1_score(y_test_class, test_pred, average="macro", zero_division=0),
    "MCC": matthews_corrcoef(y_test_class, test_pred),
}
print("\nFinal model test metrics:")
for k, v in final_test_metrics.items():
    print(f"{k:20s}: {v:.4f}")

# Bootstrap Confidence Intervals for Test Metrics

We resample the test set with replacement and compute 95% CIs for overall MCC, accuracy, F1, and per-class metrics.

In [ ]:
def bootstrap_metrics(y_true, y_pred, n_bootstrap=N_BOOTSTRAP, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    n = len(y_true)
    metrics = {
        'accuracy': [],
        'mcc': [],
        'f1_macro': [],
        'f1_class': {i: [] for i in range(3)},
    }
    for _ in range(n_bootstrap):
        idx = rng.choice(n, n, replace=True)
        y_true_b = y_true[idx]
        y_pred_b = y_pred[idx]
        metrics['accuracy'].append(accuracy_score(y_true_b, y_pred_b))
        metrics['mcc'].append(matthews_corrcoef(y_true_b, y_pred_b))
        metrics['f1_macro'].append(f1_score(y_true_b, y_pred_b, average='macro', zero_division=0))
        for c in range(3):
            y_true_c = (y_true_b == c)
            y_pred_c = (y_pred_b == c)
            metrics['f1_class'][c].append(f1_score(y_true_c, y_pred_c, zero_division=0))
    ci = {}
    for key, vals in metrics.items():
        if isinstance(vals, dict):
            ci[key] = {c: (np.percentile(vals[c], 2.5), np.percentile(vals[c], 97.5)) for c in vals}
        else:
            ci[key] = (np.percentile(vals, 2.5), np.percentile(vals, 97.5))
    return ci

ci = bootstrap_metrics(y_test_class, test_pred)
print("Bootstrap 95% CIs for test metrics:")
print(f"Accuracy : {ci['accuracy'][0]:.4f} - {ci['accuracy'][1]:.4f}")
print(f"MCC      : {ci['mcc'][0]:.4f} - {ci['mcc'][1]:.4f}")
print(f"F1 macro : {ci['f1_macro'][0]:.4f} - {ci['f1_macro'][1]:.4f}")
for c in range(3):
    print(f"F1 class {TIER_NAMES_ORDERED[c]}: {ci['f1_class'][c][0]:.4f} - {ci['f1_class'][c][1]:.4f}")


# Statistical Comparison: Best Model vs Ensemble/Stacking/LightGBM/XGBoost


In [ ]:
import numpy as np
import pandas as pd
import joblib
from scipy.stats import wilcoxon, ttest_rel
from sklearn.base import clone, BaseEstimator, ClassifierMixin
from sklearn.metrics import matthews_corrcoef
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import VarianceThreshold
import warnings
warnings.filterwarnings("ignore")

# ---- Safe ColumnSubsetWrapper ----
class ColumnSubsetWrapper(BaseEstimator, ClassifierMixin):
    def __init__(self, estimator, cols=None):
        self.estimator = estimator
        self.cols = cols

    def _select(self, X):
        if self.cols is None:
            return X
        # Keep only columns that exist in X (prevents KeyError)
        available = [c for c in self.cols if c in X.columns]
        return X[available]

    def fit(self, X, y, sample_weight=None):
        self.estimator_ = clone(self.estimator) # Clone the estimator before fitting
        X_sub = self._select(X)
        if sample_weight is not None:
            self.estimator_.fit(X_sub, y, sample_weight=sample_weight)
        else:
            self.estimator_.fit(X_sub, y)
        self.classes_ = self.estimator_.classes_ # Expose the classes_ attribute
        return self

    def predict(self, X):
        X_sub = self._select(X)
        return self.estimator_.predict(X_sub) # Use estimator_

    def predict_proba(self, X):
        X_sub = self._select(X)
        return self.estimator_.predict_proba(X_sub) # Use estimator_

# ---- ProbaAverageEnsembleClassifier ----
class ProbaAverageEnsembleClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, estimator_a=None, estimator_b=None):
        self.estimator_a = estimator_a
        self.estimator_b = estimator_b

    def fit(self, X, y, sample_weight=None):
        self.estimator_a_ = clone(self.estimator_a)
        self.estimator_b_ = clone(self.estimator_b)
        if hasattr(self.estimator_b_, 'early_stopping_rounds'):
            self.estimator_b_.set_params(early_stopping_rounds=None)
        if sample_weight is not None:
            self.estimator_a_.fit(X, y, sample_weight=sample_weight)
            self.estimator_b_.fit(X, y, sample_weight=sample_weight)
        else:
            self.estimator_a_.fit(X, y)
            self.estimator_b_.fit(X, y)
        self.classes_ = self.estimator_a_.classes_
        return self

    def predict_proba(self, X):
        return (self.estimator_a_.predict_proba(X) + self.estimator_b_.predict_proba(X)) / 2.0

    def predict(self, X):
        return self.classes_[np.argmax(self.predict_proba(X), axis=1)]

# ---- prepare_filtered_split ----
def prepare_filtered_split(train_idx_s, val_idx_s, test_idx_s, variance_threshold=0.01, corr_threshold=0.95):
    fp_cols_s = [f"FP_{i}" for i in range(hybrid_fp.shape[1])]
    X_fp = pd.DataFrame(hybrid_fp, columns=fp_cols_s)
    X_all_s = pd.concat([X_fp, desc_df_all.reset_index(drop=True)], axis=1)

    X_train_raw = X_all_s.iloc[train_idx_s].copy()
    X_val_raw = X_all_s.iloc[val_idx_s].copy()
    X_test_raw = X_all_s.iloc[test_idx_s].copy()

    medians = X_train_raw[desc_cols].median(numeric_only=True).fillna(0.0)
    X_train_raw[desc_cols] = X_train_raw[desc_cols].fillna(medians)
    X_val_raw[desc_cols] = X_val_raw[desc_cols].fillna(medians)
    X_test_raw[desc_cols] = X_test_raw[desc_cols].fillna(medians)

    selector = VarianceThreshold(threshold=variance_threshold)
    X_train_var = pd.DataFrame(selector.fit_transform(X_train_raw), columns=X_train_raw.columns[selector.get_support()])
    X_val_var = pd.DataFrame(selector.transform(X_val_raw), columns=X_train_var.columns)
    X_test_var = pd.DataFrame(selector.transform(X_test_raw), columns=X_train_var.columns)

    corr_matrix = X_train_var.corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [c for c in upper.columns if any(upper[c] > corr_threshold)]

    return X_train_var.drop(columns=to_drop), X_val_var.drop(columns=to_drop), X_test_var.drop(columns=to_drop)

# ---- Load core data ----
features = joblib.load("features.pkl")
splits = joblib.load("splits.pkl")

hybrid_fp = features["hybrid_fp"]
desc_df_all = features["desc_2d"]
desc_cols = features["desc_cols"]
y_class_full = features["y_class"]

# ---- Load models and metadata ----
# If not in memory, load from disk (you must have saved them earlier)
required_objects = [
    "best_rf", "best_xgb", "best_lgb", "logreg_baseline",
    "rf_selected_cols", "xgb_selected_cols", "lgb_selected_cols",
    "best_model_name"
]
missing = [obj for obj in required_objects if obj not in locals()]
if missing:
    print(f"Loading missing objects: {missing}")
    best_rf = joblib.load("best_rf.joblib")
    best_xgb = joblib.load("best_xgb.joblib")
    best_lgb = joblib.load("best_lgb.joblib")
    logreg_baseline = joblib.load("logreg_baseline.joblib")
    rf_selected_cols = joblib.load("rf_selected_cols.joblib")
    xgb_selected_cols = joblib.load("xgb_selected_cols.joblib")
    lgb_selected_cols = joblib.load("lgb_selected_cols.joblib")
    best_model_name = joblib.load("best_model_name.joblib")

# ---- Base models (hyperparameters only, will be wrapped later) ----
rf_base = clone(best_rf)
xgb_base = clone(best_xgb)
xgb_base.set_params(early_stopping_rounds=None)
lgb_base = clone(best_lgb)

# ---- Name map for short names ----
name_map = {
    "Logistic Regression": "Logistic Regression",
    "Random Forest Classifier": "Random Forest",
    "XGBoost Classifier": "XGBoost",
    "LightGBM Classifier": "LightGBM",
    "Ensemble (RF + XGB)": "Ensemble",
    "Stacking (RF + XGB -> LR)": "Stacking",
}

# ---- We'll rebuild models inside the loop to avoid clone issues ----
mcc_by_seed = {
    "Logistic Regression": [],
    "Random Forest": [],
    "XGBoost": [],
    "LightGBM": [],
    "Ensemble": [],
    "Stacking": []
}

for seed, (tr_s, va_s, te_s) in splits["repeated_splits"].items():
    X_tr_s, X_va_s, X_te_s = prepare_filtered_split(tr_s, va_s, te_s)
    y_tr_s, y_te_s = y_class_full[tr_s], y_class_full[te_s]
    sw_tr_s = compute_sample_weight("balanced", y_tr_s)

    # ----- Build fresh wrappers with columns that actually exist -----
    # Filter each selected list to only columns present in X_tr_s
    rf_cols_avail = [c for c in rf_selected_cols if c in X_tr_s.columns]
    xgb_cols_avail = [c for c in xgb_selected_cols if c in X_tr_s.columns]
    lgb_cols_avail = [c for c in lgb_selected_cols if c in X_tr_s.columns]

    # Wrapped models
    rf_wrapped = ColumnSubsetWrapper(clone(rf_base), cols=rf_cols_avail)
    xgb_wrapped = ColumnSubsetWrapper(clone(xgb_base), cols=xgb_cols_avail)
    lgb_wrapped = ColumnSubsetWrapper(clone(lgb_base), cols=lgb_cols_avail)

    # Ensemble (average of RF and XGB)
    ensemble_model = ProbaAverageEnsembleClassifier(
        estimator_a=clone(rf_wrapped),
        estimator_b=clone(xgb_wrapped)
    )

    # Stacking (RF + XGB -> LR)
    stacking_model = StackingClassifier(
        estimators=[
            ("rf", clone(rf_wrapped)),
            ("xgb", clone(xgb_wrapped))
        ],
        final_estimator=LogisticRegression(max_iter=5000, random_state=42),
        stack_method="predict_proba",
        passthrough=False,
        cv=3,
        n_jobs=-1
    )

    # Logistic Regression (already a pipeline, no wrapping needed)
    logreg_model = clone(logreg_baseline)

    # ----- Fit and evaluate each model -----
    # Logistic Regression (internal class_weight)
    logreg_model.fit(X_tr_s, y_tr_s)
    pred_logreg = logreg_model.predict(X_te_s)
    mcc_by_seed["Logistic Regression"].append(matthews_corrcoef(y_te_s, pred_logreg))

    # Random Forest (internal class_weight)
    rf_wrapped.fit(X_tr_s, y_tr_s)
    pred_rf = rf_wrapped.predict(X_te_s)
    mcc_by_seed["Random Forest"].append(matthews_corrcoef(y_te_s, pred_rf))

    # XGBoost (needs sample_weight)
    xgb_wrapped.fit(X_tr_s, y_tr_s, sample_weight=sw_tr_s)
    pred_xgb = xgb_wrapped.predict(X_te_s)
    mcc_by_seed["XGBoost"].append(matthews_corrcoef(y_te_s, pred_xgb))

    # LightGBM (needs sample_weight)
    lgb_wrapped.fit(X_tr_s, y_tr_s, sample_weight=sw_tr_s)
    pred_lgb = lgb_wrapped.predict(X_te_s)
    mcc_by_seed["LightGBM"].append(matthews_corrcoef(y_te_s, pred_lgb))

    # Ensemble (needs sample_weight)
    ensemble_model.fit(X_tr_s, y_tr_s, sample_weight=sw_tr_s)
    pred_ens = ensemble_model.predict(X_te_s)
    mcc_by_seed["Ensemble"].append(matthews_corrcoef(y_te_s, pred_ens))

    # Stacking (needs sample_weight)
    stacking_model.fit(X_tr_s, y_tr_s, sample_weight=sw_tr_s)
    pred_stack = stacking_model.predict(X_te_s)
    mcc_by_seed["Stacking"].append(matthews_corrcoef(y_te_s, pred_stack))

# ---- Print results ----
print("\nMCC per seed across models:")
for name, mccs in mcc_by_seed.items():
    print(f"{name:20s}: mean={np.mean(mccs):.4f}, std={np.std(mccs):.4f}")

# ---- Statistical tests ----
best_short = name_map[best_model_name]
best_mcc = mcc_by_seed[best_short]
for other in ["Ensemble", "Stacking", "LightGBM", "XGBoost"]:
    if other == best_short:
        continue
    other_mcc = mcc_by_seed[other]
    t_stat, p_val_t = ttest_rel(best_mcc, other_mcc)
    w_stat, p_val_w = wilcoxon(best_mcc, other_mcc)
    print(f"\n{best_short} vs {other}: t-test p={p_val_t:.4f}, Wilcoxon p={p_val_w:.4f}")
    print(f"  {best_short} mean+/-std: {np.mean(best_mcc):.4f}+/-{np.std(best_mcc):.4f}")
    print(f"  {other} mean+/-std: {np.mean(other_mcc):.4f}+/-{np.std(other_mcc):.4f}")

# Probability Calibration Check

We compute reliability curves and Brier scores for the best model's probabilities, using the original
validation set (before merging into the final training set) to avoid data leakage.

In [ ]:
if hasattr(best_model, "predict_proba"):
    # Filter X_val with the columns that the best_model (best_xgb) was trained on.
    # The xgb_selected_cols are available from the `xgb_optuna` cell.
    if best_model_name == "XGBoost Classifier":
        val_X_for_calibration = X_val[xgb_selected_cols]
    elif best_model_name == "Random Forest Classifier":
        val_X_for_calibration = X_val[rf_selected_cols]
    elif best_model_name == "LightGBM Classifier":
        val_X_for_calibration = X_val[lgb_selected_cols]
    elif best_model_name in ["Ensemble (RF + XGB)", "Stacking (RF + XGB -> LR)"]:
        # Ensemble and Stacking models use ColumnSubsetWrapper internally, which handles feature selection.
        # So, pass the full X_val as the wrapper will handle subsetting.
        val_X_for_calibration = X_val
    else: # e.g., Logistic Regression which uses all features (via pipeline)
        val_X_for_calibration = X_val

    val_proba = best_model.predict_proba(val_X_for_calibration)
    y_val_onehot = label_binarize(y_val_class, classes=CLASS_LABELS_ORDERED)
    brier = brier_score_loss(y_val_onehot.ravel(), val_proba.ravel())
    print(f"Brier score (validation, before feature reduction): {brier:.4f}")

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    for i, ax in enumerate(axes):
        prob_true, prob_pred = calibration_curve(y_val_class == i, val_proba[:, i], n_bins=10, strategy='quantile')
        ax.plot(prob_pred, prob_true, marker='o', label='Calibration curve')
        ax.plot([0,1], [0,1], 'k--', label='Perfect')
        ax.set_xlabel('Mean predicted probability')
        ax.set_ylabel('Fraction of positives')
        ax.set_title(f'Class {TIER_NAMES_ORDERED[i]}')
        ax.legend()
    plt.tight_layout()
    fig.savefig('calibration_curves.png', dpi=300, bbox_inches='tight')
    fig.savefig('calibration_curves.pdf', bbox_inches='tight')  # vector, for journal submission
    plt.show()
else:
    print("Model does not support predict_proba, skipping calibration check.")

# Figures: Confusion Matrix, ROC, Feature Importance, SHAP

We generate these for the final selected model (after feature reduction).

In [ ]:
cm = confusion_matrix(y_test_class, test_pred, labels=CLASS_LABELS_ORDERED)
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
im = axes[0].imshow(cm, cmap="Blues")
axes[0].set_xticks(range(3)); axes[0].set_xticklabels(TIER_NAMES_ORDERED, rotation=30)
axes[0].set_yticks(range(3)); axes[0].set_yticklabels(TIER_NAMES_ORDERED)
axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("True")
axes[0].set_title(f"Confusion matrix -- {best_model_name}")
for i in range(3):
    for j in range(3):
        axes[0].text(j, i, cm[i, j], ha="center", va="center",
                     color="white" if cm[i, j] > cm.max() / 2 else "black")

if hasattr(final_model, "predict_proba"):
    y_test_bin = label_binarize(y_test_class, classes=CLASS_LABELS_ORDERED)
    test_proba_final = final_model.predict_proba(final_test_X)
    for i, tier in enumerate(TIER_NAMES_ORDERED):
        fpr, tpr, _ = roc_curve(y_test_bin[:, i], test_proba_final[:, i])
        axes[1].plot(fpr, tpr, label=f"{tier} (AUC={auc(fpr, tpr):.2f})")
    axes[1].plot([0, 1], [0, 1], "k--", linewidth=1)
    axes[1].set_xlabel("False positive rate"); axes[1].set_ylabel("True positive rate")
    axes[1].set_title("ROC curves (one-vs-rest)")
    axes[1].legend(frameon=False, fontsize=8)
plt.tight_layout()
fig.savefig('confusion_matrix_roc.png', dpi=300, bbox_inches='tight')
fig.savefig('confusion_matrix_roc.pdf', bbox_inches='tight')  # vector, for journal submission
plt.show()
print(classification_report(y_test_class, test_pred, target_names=TIER_NAMES_ORDERED, zero_division=0))

if best_model_name in ["XGBoost Classifier", "Random Forest Classifier", "LightGBM Classifier"]:
    importances = final_model.feature_importances_
    if hasattr(final_test_X, 'columns'):
        feat_names = final_test_X.columns
    else:
        feat_names = [f"feat_{i}" for i in range(final_test_X.shape[1])]
    imp_series = pd.Series(importances, index=feat_names).sort_values(ascending=False)
    top20 = imp_series.head(20)
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.barh(top20.index[::-1], top20.values[::-1], color="#4C78A8")
    ax.set_xlabel("Importance")
    ax.set_title(f"Top 20 features, {best_model_name} (reduced set)")
    plt.tight_layout()
    fig.savefig('feature_importance_top20.png', dpi=300, bbox_inches='tight')
    fig.savefig('feature_importance_top20.pdf', bbox_inches='tight')  # vector, for journal submission
    plt.show()

    shap_sample = final_test_X.sample(min(200, final_test_X.shape[0]), random_state=RANDOM_STATE)
    explainer = shap.TreeExplainer(final_model)
    shap_values = explainer.shap_values(shap_sample)
    active_idx = TIER_TO_LABEL["Active"]
    if isinstance(shap_values, list):
        shap_vals_active = shap_values[active_idx]
    else:
        shap_vals_active = shap_values[:, :, active_idx]
    fig_shap = plt.figure(figsize=(7, 6))
    shap.summary_plot(shap_vals_active, shap_sample, show=False, max_display=20)
    plt.title(f"SHAP summary, {best_model_name} (Active class)")
    plt.tight_layout()
    fig_shap.savefig('shap_summary_active.png', dpi=300, bbox_inches='tight')
    fig_shap.savefig('shap_summary_active.pdf', bbox_inches='tight')  # vector, for journal submission
    plt.show()
else:
    print("Best model is not tree-based; skipping feature importance and SHAP.")

# Y-Randomization Validation

We permute the training labels and refit the final model N times to see if the true MCC is significantly
better than random.


In [ ]:

if final_bias is not None:
    def predict_fn(m, X):
        return np.argmax(m.predict_proba(X) * np.exp(final_bias), axis=1)
else:
    def predict_fn(m, X):
        return m.predict(X)

final_uses_internal_balance = getattr(final_model, "class_weight", None) is not None
final_sw_for_yrand = None if final_uses_internal_balance else compute_sample_weight("balanced", final_train_y)

def y_randomization_test(model, X_tr, y_tr, X_te, y_te, sample_weight_tr=None, true_mcc=None,
                          predict_fn=None, n_permutations=N_YRAND_PERMUTATIONS, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    if predict_fn is None:
        predict_fn = lambda m, X: m.predict(X)
    if true_mcc is None:
        true_mcc = matthews_corrcoef(y_te, predict_fn(model, X_te))
    permuted_mccs = []
    print(f"Running {n_permutations} permutations...")
    for i in tqdm(range(n_permutations)):
        y_perm = rng.permutation(y_tr)
        perm_model = clone(model)
        if hasattr(perm_model, 'early_stopping_rounds'):
            perm_model.set_params(early_stopping_rounds=None)
        if sample_weight_tr is not None:
            perm_model.fit(X_tr, y_perm, sample_weight=sample_weight_tr)
        else:
            perm_model.fit(X_tr, y_perm)
        permuted_mccs.append(matthews_corrcoef(y_te, predict_fn(perm_model, X_te)))
    permuted_mccs = np.array(permuted_mccs)
    p_value = (np.sum(permuted_mccs >= true_mcc) + 1) / (n_permutations + 1)
    return true_mcc, permuted_mccs, p_value

print("Y-randomization test on final model...")
true_mcc = final_test_metrics["MCC"]
_, permuted_mccs, p_val = y_randomization_test(
    final_model, final_train_X, final_train_y, final_test_X, y_test_class,
    sample_weight_tr=final_sw_for_yrand,
    true_mcc=true_mcc,
    predict_fn=predict_fn,
    n_permutations=N_YRAND_PERMUTATIONS
)
print(f"\nTrue MCC: {true_mcc:.4f}")
print(f"Permuted MCC mean: {np.mean(permuted_mccs):.4f} +/- {np.std(permuted_mccs):.4f}")
print(f"Empirical p-value: {p_val:.4f}")
if p_val < 0.05:
    print("The model is statistically significant (p < 0.05).")
else:
    print("The model is NOT statistically significant (p >= 0.05).")

fig_yrand = plt.figure(figsize=(6,4))
plt.hist(permuted_mccs, bins=30, color="#9ECAE1", edgecolor='white', label='Y-randomized MCC')
plt.axvline(true_mcc, color='#B23A48', linewidth=2, label=f'True MCC = {true_mcc:.3f}')
plt.xlabel("MCC"); plt.ylabel("Count")
plt.title(f"Y-randomization ({N_YRAND_PERMUTATIONS} permutations)")
plt.legend(frameon=False)
plt.tight_layout()
fig_yrand.savefig('y_randomization.png', dpi=300, bbox_inches='tight')
fig_yrand.savefig('y_randomization.pdf', bbox_inches='tight')  # vector, for journal submission
plt.show()

# Applicability Domain (Tanimoto)



In [ ]:
if best_model_name in ["XGBoost Classifier", "Random Forest Classifier", "LightGBM Classifier"]:
    ad_train_idx = final_train_idx
else:
    ad_train_idx = train_idx

train_fps_for_ad = features["ecfp4"][ad_train_idx]
test_fps_for_ad = features["ecfp4"][test_idx]
test_ad_similarity = nearest_training_tanimoto(test_fps_for_ad, train_fps_for_ad)

clf_ad_df = pd.DataFrame({
    "canonical_smiles": [smiles_all[i] for i in test_idx],
    "true_tier": [LABEL_TO_TIER[c] for c in y_test_class],
    "predicted_tier": [LABEL_TO_TIER[c] for c in test_pred],
    "correct": (test_pred == y_test_class),
    "nearest_training_tanimoto": test_ad_similarity,
})
clf_ad_df["within_applicability_domain"] = clf_ad_df["nearest_training_tanimoto"] >= AD_TANIMOTO_THRESHOLD
clf_ad_accuracy_by_domain = clf_ad_df.groupby("within_applicability_domain")["correct"].agg(["mean", "count"])
clf_ad_accuracy_by_domain.columns = ["Accuracy", "N"]
print("Classifier accuracy inside vs. outside the applicability domain (1-NN):")
print(clf_ad_accuracy_by_domain)

clf_ad_df[f"mean_top{AD_KNN_K}_training_tanimoto"] = topk_mean_training_tanimoto(
    test_fps_for_ad, train_fps_for_ad, k=AD_KNN_K
)
clf_ad_df["within_applicability_domain_knn"] = clf_ad_df[f"mean_top{AD_KNN_K}_training_tanimoto"] >= AD_TANIMOTO_THRESHOLD
clf_ad_accuracy_by_domain_knn = clf_ad_df.groupby("within_applicability_domain_knn")["correct"].agg(["mean", "count"])
clf_ad_accuracy_by_domain_knn.columns = ["Accuracy", "N"]
print(f"\nClassifier accuracy inside vs. outside the k-NN (k={AD_KNN_K}) applicability domain:")
print(clf_ad_accuracy_by_domain_knn)

agreement = (clf_ad_df["within_applicability_domain"] == clf_ad_df["within_applicability_domain_knn"]).mean()
print(f"\n1-NN and top-{AD_KNN_K} domain flags agree on {agreement:.1%} of test compounds.")


# Save `classifier.joblib`

Bundle the final model and all relevant metadata. Note the custom class note for unpickling.

In [ ]:
classifier_bundle = {
    "model": final_model,
    "model_name": best_model_name,
    "feature_columns": list(final_test_X.columns) if hasattr(final_test_X, "columns") else None,
    "decision_bias": final_bias,  # None unless best model is XGBoost/LightGBM (bug fix #2)
    "descriptor_medians": splits["descriptor_medians"],
    "correlation_dropped_columns": splits["correlation_dropped_columns"],
    "tier_to_label": TIER_TO_LABEL,
    "label_to_tier": LABEL_TO_TIER,
    "class_labels_ordered": CLASS_LABELS_ORDERED,
    "tier_names_ordered": TIER_NAMES_ORDERED,
    "active_pic50_cutoff": ACTIVE_PIC50_CUTOFF,
    "inactive_pic50_cutoff": INACTIVE_PIC50_CUTOFF,
    "ad_tanimoto_threshold": AD_TANIMOTO_THRESHOLD,
    "ad_knn_k": AD_KNN_K,
    "train_ecfp4": features["ecfp4"][ad_train_idx],  # matches what final_model was actually trained on
    "test_metrics": final_test_metrics,
    "validation_selection": validation_selection,
    "y_randomization": {
        "true_mcc": true_mcc,
        "permuted_mcc": permuted_mccs,
        "p_value": p_val,
        "n_permutations": N_YRAND_PERMUTATIONS,
    },
    "note": ("Final model with feature reduction and out-of-fold re-tuned threshold. "
             "Apply predict_proba(X) * exp(decision_bias) then argmax if decision_bias is not None; "
             "otherwise use predict(X) directly. If the best model is 'Ensemble (RF + XGB)', you must "
             "redefine ProbaAverageEnsembleClassifier before loading.")
}

joblib.dump(classifier_bundle, "classifier.joblib")
print(f"Saved classifier.joblib -- best model: {best_model_name}")

try:
    from google.colab import files
    files.download("classifier.joblib")
except Exception:
    print("Not in Colab or download failed. Retrieve manually.")


In [ ]:
# Save correctly-classified test compounds, split by tier, into separate CSVs

# Confidence = probability the model assigned to its predicted class
test_proba = final_model.predict_proba(final_test_X)
model_classes = list(final_model.classes_) if hasattr(final_model, "classes_") else list(range(test_proba.shape[1]))
pred_col_idx = [model_classes.index(p) for p in test_pred]
confidence = test_proba[np.arange(len(test_pred)), pred_col_idx]

high_acc_df = clf_ad_df.copy()
high_acc_df["confidence"] = confidence

# Keep only compounds where predicted tier == true tier
correct_df = high_acc_df[high_acc_df["correct"]].reset_index(drop=True)

for tier in TIER_NAMES_ORDERED:
    tier_df = correct_df[correct_df["true_tier"] == tier]
    filename = f"high_accuracy_{tier.lower()}_compounds.csv"
    tier_df.to_csv(filename, index=False)
    print(f"Saved {len(tier_df)} {tier} compounds -> {filename}")